# Part 3, step 2 (clean) – differential expression, grouping, enrichment (tutorial 3-4 to 3-7)
Run each cell with **Shift+Enter**; plots appear below the cell.

Comparisons are done **within one cohort only** (README):

| Name | Cohort | Case vs control | Covariate |
|---|---|---|---|
| `aging` | Aging | 21 mo vs 3 mo | sex |
| `diet` | Dietary Intervention | aDR vs AL | – |
| `ymp` | Young Plasma Injection | YMP vs PBS | – |

Same analysis as `03_part3_step2.ipynb`, but **all samples flagged as bad in step 1 / 1b are excluded**:

| Sample | Group | Why |
|---|---|---|
| SRR21355389 | aging 21mo | blood contamination (hemoglobin, neutrophil genes); dominates PC1 of the aging cohort |
| SRR23914572 | AL (mouse 572) | outlier of the diet cohort; alone on PC1, AL samples only cluster without it |

This leaves 25 samples: aging 3mo 4 / 18mo 4 / 21mo 3, AL 3 vs aDR 5, PBS 3 vs YMP 3. Results are written to `part3_output/clean/`, so the files of the first step 2 run are kept.

In [ ]:
options(jupyter.display_mimetypes = c("text/plain", "text/html"))

## Setup

In [ ]:
library(tidyverse)
library(ggrepel)
library(pbapply)
library(DESeq2)

options(repr.plot.width = 8, repr.plot.height = 5.5)   # plot size in the notebook
# White plot background (otherwise VS Code dark theme shows through)
setHook("before.plot.new", function() par(bg = "white"))
theme_update(plot.background = element_rect(fill = "white", colour = NA))

base <- "~/Documents/claude/Claude/Systems Genomics/RNAseq_aging/for_teammates"
out  <- "~/Documents/claude/Claude/Systems Genomics/part3_output/clean"
dir.create(out, recursive = TRUE, showWarnings = FALSE)

read_table <- function(f) as.matrix(read.csv(file.path(base, f), row.names = 1, check.names = FALSE))
expr   <- read_table("rsem_tpm.csv.gz")               # TPM
counts <- read_table("rsem_expected_counts.csv.gz")   # estimated reads (for DESeq2)
efflen <- read_table("rsem_effective_length.csv.gz")  # gene length per sample (for DESeq2)

meta <- read.csv(file.path(base, "samples.csv"))
rownames(meta) <- meta$SRR

# Samples flagged as bad in step 1 / 1b
excluded <- c("SRR21355389",   # aging 21mo, blood contamination
              "SRR23914572")   # AL 572, outlier of the diet cohort
meta <- meta[!meta$SRR %in% excluded, ]
expr   <- expr[, meta$SRR]
counts <- counts[, meta$SRR]
efflen <- efflen[, meta$SRR]

meta_genes <- read.csv(file.path(base, "genes_M39.csv"))
meta_genes <- data.frame(gene_id = rownames(expr)) %>%
  left_join(meta_genes, by = "gene_id") %>%
  mutate(ensembl = sub("\\..*", "", gene_id))          # Ensembl id without version (.6)
sym <- setNames(meta_genes$symbol, meta_genes$gene_id)

table(meta$group)

In [ ]:
# The three comparisons
comparisons <- list(
  aging = list(cohort = "Aging",                  ctrl = "aging 3mo", case = "aging 21mo", covar = "sex"),
  diet  = list(cohort = "Dietary Intervention",   ctrl = "AL",        case = "aDR",        covar = NULL),
  ymp   = list(cohort = "Young Plasma Injection", ctrl = "PBS",       case = "YMP",        covar = NULL)
)

# Samples of one comparison
get_samples <- function(cmp) meta[meta$cohort == cmp$cohort & meta$group %in% c(cmp$ctrl, cmp$case), ]

# DE thresholds used throughout
padj_cut <- 0.1         # adjusted p-value
lfc_cut  <- log2(1.5)   # at least 1.5-fold change

sapply(comparisons, function(cmp) table(get_samples(cmp)$group))

## 3-4 Differential expression analysis
### ANCOVA on one gene
Full model `y ~ group + sex` vs null model `y ~ sex`; `anova` tests whether adding the group improves the fit. Example: *C4b*, a complement gene known to increase in the aging brain.

In [ ]:
m <- get_samples(comparisons$aging)
g <- meta_genes$gene_id[which(meta_genes$symbol == "C4b")]

dat <- data.frame(y = log1p(expr[g, m$SRR]), group = m$group, sex = m$sex)
m1 <- lm(y ~ group + sex, data = dat)
m0 <- lm(y ~ sex, data = dat)
anova(m1, m0)$Pr[2]

options(repr.plot.width = 5, repr.plot.height = 4)
ggplot(dat, aes(group, y, colour = sex)) +
  geom_point(size = 3, position = position_jitter(width = 0.1, seed = 1)) +
  labs(y = "log(TPM + 1)", title = "C4b") +
  theme_bw()

### ANCOVA on all expressed genes
The tutorial's `DE_test`, adapted to a case-control comparison: the fold change is case vs control (so it can be negative on the log scale), and p-values are corrected with Benjamini–Hochberg (BH; less strict than Bonferroni, which is too conservative with 3–5 samples per group). Genes are filtered within the samples of each comparison.

In [ ]:
DE_test <- function(expr, cond, ctrl, covar = NULL, padj_method = "BH") {
  pval_fc <- data.frame(t(pbapply(expr, 1, function(e) {
    dat <- data.frame(y = log1p(e), cond = cond)
    if (!is.null(covar))
      dat <- data.frame(dat, covar)

    m1 <- lm(y ~ ., data = dat)
    m0 <- lm(y ~ . - cond, data = dat)
    pval <- anova(m1, m0)$Pr[2]

    avgs <- tapply(log1p(e), cond, mean)
    fc <- exp(avgs[names(avgs) != ctrl] - avgs[ctrl])
    c(pval = unname(pval), fc = unname(fc))
  })), row.names = rownames(expr))
  pval_fc$padj   <- p.adjust(pval_fc$pval, method = padj_method)
  pval_fc$log2FC <- log2(pval_fc$fc)
  pval_fc
}

run_ancova <- function(cmp) {
  m <- get_samples(cmp)
  e <- expr[, m$SRR]
  expressed <- rowMeans(e > 0) >= 0.5 | rowMeans(e) >= 1
  covar <- if (is.null(cmp$covar)) NULL else m[, cmp$covar, drop = FALSE]
  DE_test(e[expressed, ], cond = m$group, ctrl = cmp$ctrl, covar = covar) %>%
    rownames_to_column("gene") %>%
    mutate(symbol = sym[gene])
}

res_DE <- lapply(comparisons, run_ancova)   # takes ~1-2 minutes
sapply(res_DE, function(r) c(genes = nrow(r),
                             DEGs  = sum(r$padj < padj_cut & abs(r$log2FC) > lfc_cut)))

### Volcano plots (ANCOVA)

In [ ]:
plot_volcano <- function(res, title, lfc = "log2FC", p = "padj") {
  res$DE <- !is.na(res[[p]]) & res[[p]] < padj_cut & abs(res[[lfc]]) > lfc_cut
  top <- rank(res[[p]], ties.method = "first") <= 20         # label the 20 most significant DEGs
  res$label <- ifelse(res$DE & top, res$symbol, NA)
  ggplot(res, aes(.data[[lfc]], -log10(.data[[p]]), colour = DE, label = label)) +
    geom_point(size = 0.8) +
    geom_text_repel(size = 3, max.overlaps = 30, na.rm = TRUE, show.legend = FALSE) +
    geom_vline(xintercept = c(-lfc_cut, lfc_cut), col = "#303030", linetype = "dotted") +
    geom_hline(yintercept = -log10(padj_cut), col = "#303030", linetype = "dotted") +
    scale_colour_manual(values = c(`FALSE` = "#909090", `TRUE` = "red")) +
    labs(x = "log2 fold change (case vs control)", y = "-log10(padj)",
         title = paste0(title, " – ", sum(res$DE), " DEGs")) +
    theme_bw()
}

options(repr.plot.width = 7, repr.plot.height = 5)
for (n in names(comparisons)) {
  cmp <- comparisons[[n]]
  print(plot_volcano(res_DE[[n]], paste0("ANCOVA: ", cmp$case, " vs ", cmp$ctrl)))
}

### DESeq2
DESeq2 needs counts. `tximport` would sum RSEM's transcript files; the teammates' tables already contain the gene-level output, so we hand it over in the same shape `tximport` produces (counts + TPM + effective length). DESeq2 rounds the counts and uses the lengths to correct for gene length.

As in the tutorial: likelihood ratio test (LRT), full model with the group vs reduced model without it, on the same expressed genes as the ANCOVA.

In [ ]:
run_deseq2 <- function(cmp, genes) {
  m <- get_samples(cmp)
  m$group <- factor(m$group, levels = c(cmp$ctrl, cmp$case))   # control first = reference
  m$sex   <- factor(m$sex)

  len <- efflen[, m$SRR]
  len[len == 0] <- 1                                           # DESeq2 needs length > 0
  txi <- list(abundance = expr[, m$SRR], counts = counts[, m$SRR],
              length = len, countsFromAbundance = "no")

  design  <- if (is.null(cmp$covar)) ~ group else ~ sex + group
  reduced <- if (is.null(cmp$covar)) ~ 1     else ~ sex
  dds <- DESeqDataSetFromTximport(txi, colData = m, design = design)
  dds <- DESeq(dds[genes, ], test = "LRT", reduced = reduced, quiet = TRUE)

  as.data.frame(results(dds)) %>%
    rownames_to_column("gene") %>%
    mutate(symbol = sym[gene])
}

res_DESeq2 <- lapply(names(comparisons), function(n) run_deseq2(comparisons[[n]], res_DE[[n]]$gene))
names(res_DESeq2) <- names(comparisons)

sapply(res_DESeq2, function(r) c(genes = nrow(r),
                                 DEGs  = sum(r$padj < padj_cut & abs(r$log2FoldChange) > lfc_cut, na.rm = TRUE)))

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5)
for (n in names(comparisons)) {
  cmp <- comparisons[[n]]
  print(plot_volcano(res_DESeq2[[n]], paste0("DESeq2: ", cmp$case, " vs ", cmp$ctrl),
                     lfc = "log2FoldChange"))
}

### Compare DESeq2 with the ANCOVA

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)
layout(matrix(1:6, nrow = 2))
for (n in names(comparisons)) {
  a <- res_DE[[n]]
  d <- res_DESeq2[[n]][match(a$gene, res_DESeq2[[n]]$gene), ]
  r <- cor(d$pvalue, a$pval, method = "spearman", use = "complete.obs")
  plot(-log10(d$pvalue), -log10(a$pval), pch = 16, cex = 0.4,
       xlab = "-log10(pval DESeq2)", ylab = "-log10(pval ANCOVA)",
       main = paste0(n, " (Spearman r = ", round(r, 2), ")"))
  abline(0, 1, col = "red")
  smoothScatter(-log10(d$pvalue), -log10(a$pval),
                xlab = "-log10(pval DESeq2)", ylab = "-log10(pval ANCOVA)", main = n)
}

In [ ]:
# DEG overlap: rows = DESeq2, columns = ANCOVA (padj < 0.1, any fold change)
for (n in names(comparisons)) {
  a <- res_DE[[n]]
  d <- res_DESeq2[[n]][match(a$gene, res_DESeq2[[n]]$gene), ]
  cat("\n", n, "\n")
  print(table(DESeq2 = d$padj < padj_cut, ANCOVA = a$padj < padj_cut, useNA = "ifany"))
}

# Save the DESeq2 tables
for (n in names(comparisons))
  write.csv(res_DESeq2[[n]], file.path(out, paste0("DESeq2_", n, ".csv")), row.names = FALSE)

## 3-5 Grouping of the identified DEGs
### Case-control: up- and down-regulated
From here on the DESeq2 results are used (more sensitive, see the comparison above).

In [ ]:
degs <- lapply(res_DESeq2, function(r) {
  r <- r[!is.na(r$padj) & r$padj < padj_cut & abs(r$log2FoldChange) > lfc_cut, ]
  list(up = r$gene[r$log2FoldChange > 0], down = r$gene[r$log2FoldChange < 0])
})
sapply(degs, lengths)

In [ ]:
# Expression of the up/down DEGs in each sample (log TPM, scaled per gene)
options(repr.plot.width = 12, repr.plot.height = 4)
layout(matrix(1:6, nrow = 1))
par(mar = c(7, 3, 3, 1))
for (n in names(comparisons)) {
  m <- get_samples(comparisons[[n]])
  m <- m[order(m$group != comparisons[[n]]$ctrl), ]          # controls first
  for (dir in c("up", "down")) {
    g <- degs[[n]][[dir]]
    if (length(g) < 2) { plot.new(); title(paste(n, dir, "(< 2 genes)")); next }
    scaled <- t(scale(t(log1p(expr[g, m$SRR]))))
    boxplot(scaled, names = m$group, las = 2, col = ifelse(m$group == comparisons[[n]]$ctrl, "grey80", "salmon"),
            main = paste0(n, " ", dir, " (", length(g), ")"))
  }
}

In [ ]:
# Heatmap of all DEGs per comparison (rows = genes, columns = samples)
library(gplots)
library(viridis)

options(repr.plot.width = 7, repr.plot.height = 7)
for (n in names(comparisons)) {
  g <- unlist(degs[[n]])
  if (length(g) < 2) { cat(n, ": fewer than 2 DEGs, no heatmap\n"); next }
  m <- get_samples(comparisons[[n]])
  heatmap.2(log1p(expr[g, m$SRR]), scale = "row", trace = "none", col = viridis,
            labRow = if (length(g) <= 60) sym[g] else NA, labCol = m$group,
            ColSideColors = ifelse(m$group == comparisons[[n]]$ctrl, "grey60", "salmon"),
            margins = c(8, 6), main = paste(n, "DEGs"))
}

### Multi-condition: all three ages (3 / 18 / 21 months)
Like the tutorial's layers: one LRT across all ages (`~ sex + group` vs `~ sex`), then group the DEGs by the age with the highest expression, and by hierarchical clustering.

In [ ]:
m_age <- meta[meta$cohort == "Aging", ]
m_age$group <- factor(m_age$group, levels = c("aging 3mo", "aging 18mo", "aging 21mo"))
m_age$sex   <- factor(m_age$sex)

e <- expr[, m_age$SRR]
expressed_age <- rowMeans(e > 0) >= 0.5 | rowMeans(e) >= 1

len <- efflen[, m_age$SRR]
len[len == 0] <- 1
txi <- list(abundance = e, counts = counts[, m_age$SRR], length = len, countsFromAbundance = "no")
dds_age <- DESeqDataSetFromTximport(txi, colData = m_age, design = ~ sex + group)
dds_age <- DESeq(dds_age[expressed_age, ], test = "LRT", reduced = ~ sex, quiet = TRUE)
res_age <- as.data.frame(results(dds_age)) %>% rownames_to_column("gene") %>% mutate(symbol = sym[gene])

DEG_age <- res_age$gene[!is.na(res_age$padj) & res_age$padj < padj_cut]
length(DEG_age)

In [ ]:
# Average expression per age, then: at which age is each DEG highest?
avg_age <- sapply(levels(m_age$group), function(a) rowMeans(expr[, m_age$SRR[m_age$group == a]]))
max_age_DEG <- setNames(colnames(avg_age)[apply(avg_age[DEG_age, ], 1, which.max)], DEG_age)
table(max_age_DEG)

scaled_list <- lapply(split(names(max_age_DEG), max_age_DEG), function(x) t(scale(t(avg_age[x, , drop = FALSE]))))
options(repr.plot.width = 10, repr.plot.height = 4)
layout(matrix(1:3, nrow = 1))
par(mar = c(4, 3, 3, 1))
for (a in levels(m_age$group))
  if (!is.null(scaled_list[[a]]))
    boxplot(scaled_list[[a]], names = c("3mo", "18mo", "21mo"),
            main = paste0("highest at ", a, " (", nrow(scaled_list[[a]]), ")"))

In [ ]:
# Hierarchical clustering of the DEGs. With only 3 ages, correlating the 3 averages is
# too coarse, so genes are compared across the individual samples (scaled log TPM)
scaled_samples <- t(scale(t(log1p(expr[DEG_age, m_age$SRR[order(m_age$group)]]))))
corr_DEG <- cor(t(scaled_samples))
hcl_DEG  <- hclust(as.dist(1 - corr_DEG), method = "complete")

n_cl   <- 4                                      # number of clusters: try other values
cl_DEG <- cutree(hcl_DEG, k = n_cl)
table(cl_DEG)

options(repr.plot.width = 7, repr.plot.height = 7)
heatmap.2(corr_DEG, Rowv = as.dendrogram(hcl_DEG), Colv = as.dendrogram(hcl_DEG),
          trace = "none", scale = "none", labRow = NA, labCol = NA, col = viridis,
          ColSideColors = rainbow(n_cl)[cl_DEG], main = "Aging DEGs: gene-gene correlation")

In [ ]:
# Average pattern of each cluster across ages, in dendrogram order
options(repr.plot.width = 12, repr.plot.height = 4)
layout(matrix(1:n_cl, nrow = 1))
par(mar = c(4, 3, 3, 1))
for (cl in unique(cl_DEG[hcl_DEG$order])) {
  g <- names(cl_DEG)[cl_DEG == cl]
  boxplot(t(scale(t(avg_age[g, , drop = FALSE]))), names = c("3mo", "18mo", "21mo"),
          border = rainbow(n_cl)[cl], main = paste0("cluster ", cl, " (", length(g), ")"))
}

## 3-6 Making sense of the genes
### Gene lists for DAVID (frequency-based enrichment)
DAVID takes Ensembl ids **without** the version suffix. Upload a gene list, and as **background** the expressed genes of that comparison (tutorial: this is critical).

In [ ]:
david <- file.path(out, "david")
dir.create(david, showWarnings = FALSE)
to_ensembl <- setNames(meta_genes$ensembl, meta_genes$gene_id)

for (n in names(comparisons)) {
  for (dir in c("up", "down"))
    writeLines(to_ensembl[degs[[n]][[dir]]], file.path(david, paste0(n, "_", dir, ".txt")))
  writeLines(to_ensembl[res_DE[[n]]$gene], file.path(david, paste0(n, "_background.txt")))
}
for (cl in sort(unique(cl_DEG)))
  writeLines(to_ensembl[names(cl_DEG)[cl_DEG == cl]], file.path(david, paste0("aging_cluster", cl, ".txt")))
writeLines(to_ensembl[res_age$gene], file.path(david, "aging_all_ages_background.txt"))

list.files(david)

### GSEA with fgsea (rank-based enrichment)
Every gene gets the score sign(log2FC) × −log10(p) from DESeq2; GSEA checks whether a gene set sits at the top (up in case) or bottom (down in case) of the ranking. Gene sets from MSigDB: **Hallmark** (50 broad processes) and **C8** (cell type signatures, as in the tutorial).

In [ ]:
library(msigdbr)
library(fgsea)

get_sets <- function(collection) {
  gs <- msigdbr(species = "Mus musculus", collection = collection)
  tapply(gs$ensembl_gene, gs$gs_name, function(x) unique(x))
}
genesets <- list(hallmark = get_sets("H"), celltype = get_sets("C8"))
lengths(genesets)

gene_scores <- function(r) {
  r <- r[!is.na(r$pvalue), ]
  s <- setNames(sign(r$log2FoldChange) * -log10(r$pvalue), to_ensembl[r$gene])
  sort(s[!duplicated(names(s))], decreasing = TRUE)
}
scores <- lapply(res_DESeq2, gene_scores)

set.seed(42)
gsea <- lapply(scores, function(s)
  lapply(genesets, function(sets) fgsea(pathways = sets, stats = s, minSize = 15, maxSize = 500)))

In [ ]:
# Hallmark: significant sets per comparison (padj < 0.1); NES > 0 = up in case
for (n in names(comparisons)) {
  cat("\n=====", n, "=====\n")
  res <- gsea[[n]]$hallmark[order(NES, decreasing = TRUE)]
  print(res[padj < padj_cut, .(pathway, NES = round(NES, 2), padj = signif(padj, 2), size)])
}

In [ ]:
# Hallmark NES of all three comparisons side by side (* = padj < 0.1)
nes <- bind_rows(lapply(names(gsea), function(n)
  gsea[[n]]$hallmark %>% as_tibble() %>% transmute(comparison = n, pathway, NES, padj)))
keep <- nes %>% group_by(pathway) %>% filter(any(padj < padj_cut)) %>% pull(pathway) %>% unique()

options(repr.plot.width = 9, repr.plot.height = max(3, 0.3 * length(keep) + 1.5))
nes %>%
  filter(pathway %in% keep) %>%
  mutate(pathway = sub("HALLMARK_", "", pathway),
         comparison = factor(comparison, levels = names(comparisons))) %>%
  ggplot(aes(comparison, reorder(pathway, NES), fill = NES)) +
  geom_tile() +
  geom_text(aes(label = ifelse(padj < padj_cut, "*", "")), size = 6, vjust = 0.75) +
  scale_fill_gradient2(low = "steelblue", mid = "white", high = "firebrick") +
  labs(x = NULL, y = NULL, title = "Hallmark gene sets (GSEA)") +
  theme_bw()

In [ ]:
# Cell type signatures (C8): top 10 up and down for each comparison
for (n in names(comparisons)) {
  res <- gsea[[n]]$celltype
  cat("\n=====", n, ": up in", comparisons[[n]]$case, "=====\n")
  print(res[order(NES, decreasing = TRUE)][1:10, .(pathway, NES = round(NES, 2), padj = signif(padj, 2))])
  cat("\n=====", n, ": down in", comparisons[[n]]$case, "=====\n")
  print(res[order(NES)][1:10, .(pathway, NES = round(NES, 2), padj = signif(padj, 2))])
}

In [ ]:
# Enrichment plot of the strongest Hallmark set in the aging comparison
top <- gsea$aging$hallmark[order(pval)]$pathway[1]
options(repr.plot.width = 7, repr.plot.height = 4)
plotEnrichment(genesets$hallmark[[top]], scores$aging) +
  labs(title = paste("aging 21 vs 3 mo:", top))

## 3-7 Other analysis
### Cell type composition (marker genes)
Real deconvolution (CIBERSORTx, quadratic programming) needs a single-cell reference of the cerebellum, which we do not have here. A simpler estimate: for each cell type, the mean log(TPM + 1) of a few well-known marker genes. A change in this score between groups suggests a change in how much of that cell type is in the tissue (or a change in its marker expression; the score cannot tell the two apart).

Only the 25 clean samples are used here.

In [ ]:
markers <- list(
  "Granule cells"      = c("Gabra6", "Cbln3", "Neurod1", "Pax6"),
  "Purkinje cells"     = c("Pcp2", "Calb1", "Car8", "Pcp4"),
  "Bergmann glia/astro"= c("Gdf10", "Aqp4", "Slc1a3", "Aldh1l1"),
  "Oligodendrocytes"   = c("Mbp", "Plp1", "Mog", "Mobp"),
  "OPC"                = c("Pdgfra", "Cspg4", "Vcan"),
  "Microglia"          = c("P2ry12", "Cx3cr1", "Tmem119", "Csf1r"),
  "Endothelial"        = c("Cldn5", "Flt1", "Pecam1"),
  "Red blood cells"    = c("Hba-a1", "Hbb-bs", "Alas2"),
  "Neutrophils"        = c("S100a8", "S100a9", "Ngp")
)

# The 25 clean samples
expr_all <- expr
meta_all <- meta

sym_to_id <- setNames(meta_genes$gene_id, meta_genes$symbol)
missing <- setdiff(unlist(markers), names(sym_to_id))
if (length(missing) > 0) cat("not found:", missing, "\n")

celltype_score <- sapply(markers, function(g) colMeans(log1p(expr_all[sym_to_id[intersect(g, names(sym_to_id))], , drop = FALSE])))
round(celltype_score, 2)

In [ ]:
group_order <- c("aging 3mo", "aging 18mo", "aging 21mo", "AL", "aDR", "PBS", "YMP")
score_long <- data.frame(meta_all, celltype_score, check.names = FALSE) %>%
  pivot_longer(names(markers), names_to = "cell_type", values_to = "score") %>%
  mutate(group = factor(group, levels = group_order),
         cell_type = factor(cell_type, levels = names(markers)))

options(repr.plot.width = 13, repr.plot.height = 9)
ggplot(score_long, aes(group, score, colour = cohort)) +
  geom_boxplot(outlier.shape = NA, colour = "grey60") +
  geom_point(size = 2, position = position_jitter(width = 0.15, seed = 1)) +
  facet_wrap(~ cell_type, scales = "free_y") +
  labs(x = NULL, y = "mean log(TPM + 1) of markers", title = "Cell type marker scores") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), legend.position = "bottom")

In [ ]:
# Test each marker score per comparison (same models as the ANCOVA)
score_test <- bind_rows(lapply(names(comparisons), function(n) {
  cmp <- comparisons[[n]]
  m <- get_samples(cmp)
  bind_rows(lapply(names(markers), function(ct) {
    dat <- data.frame(y = celltype_score[m$SRR, ct], group = factor(m$group, levels = c(cmp$ctrl, cmp$case)), sex = m$sex)
    f1 <- if (is.null(cmp$covar)) y ~ group else y ~ sex + group
    f0 <- if (is.null(cmp$covar)) y ~ 1     else y ~ sex
    data.frame(comparison = n, cell_type = ct,
               diff = coef(lm(f1, dat))[[paste0("group", cmp$case)]],   # case - control
               pval = anova(lm(f1, dat), lm(f0, dat))$Pr[2])
  }))
})) %>%
  mutate(padj = p.adjust(pval, method = "BH"))

options(repr.plot.width = 7, repr.plot.height = 5)
ggplot(score_test, aes(factor(comparison, levels = names(comparisons)),
                       factor(cell_type, levels = rev(names(markers))), fill = diff)) +
  geom_tile() +
  geom_text(aes(label = ifelse(padj < padj_cut, "*", "")), size = 6, vjust = 0.75) +
  scale_fill_gradient2(low = "steelblue", mid = "white", high = "firebrick") +
  labs(x = NULL, y = NULL, fill = "case - control",
       title = "Marker score change (* = padj < 0.1)") +
  theme_bw()

### Alternative splicing: isoform usage
rMATS needs the read alignments (BAM files), which are not here. But RSEM also estimated, per sample, how much each **isoform** (transcript) contributes to its gene (`IsoPct`, in %). These `*.isoforms.results` files are in your own Euler output (`euler_results/rsem_trimmed`, same RSEM run as the teammates' tables). A change in an isoform's share between groups, while the gene itself may stay the same, is called **differential transcript usage** (DTU), a sign of alternative splicing (or alternative start/end).

Caution: isoform shares from short reads are uncertain, especially for low-expressed genes, so only isoforms with TPM ≥ 1 and a share ≥ 5 % are tested.

In [ ]:
library(data.table)
iso_dir <- "~/Documents/claude/Claude/Systems Genomics/euler_results/rsem_trimmed"

iso <- lapply(meta$SRR, function(s)
  fread(file.path(iso_dir, s, paste0(s, ".isoforms.results")), select = c("transcript_id", "gene_id", "TPM", "IsoPct")))
stopifnot(all(sapply(iso, function(x) identical(x$transcript_id, iso[[1]]$transcript_id))))

tx2gene <- data.frame(transcript_id = iso[[1]]$transcript_id, gene_id = iso[[1]]$gene_id)
iso_tpm <- sapply(iso, `[[`, "TPM");    dimnames(iso_tpm) <- list(tx2gene$transcript_id, meta$SRR)
iso_pct <- sapply(iso, `[[`, "IsoPct"); dimnames(iso_pct) <- list(tx2gene$transcript_id, meta$SRR)
rm(iso)
dim(iso_pct)   # transcripts x samples

In [ ]:
dtu_test <- function(cmp, genes) {
  m <- get_samples(cmp)
  tx <- tx2gene$gene_id %in% genes &
        rowMeans(iso_tpm[, m$SRR]) >= 1 & rowMeans(iso_pct[, m$SRR]) >= 5
  multi <- names(which(table(tx2gene$gene_id[tx]) >= 2))     # genes with >= 2 such isoforms
  tx <- tx & tx2gene$gene_id %in% multi

  covar <- if (is.null(cmp$covar)) NULL else m[, cmp$covar, drop = FALSE]
  res <- data.frame(t(pbapply(iso_pct[tx, m$SRR], 1, function(p) {
    dat <- data.frame(y = p, cond = m$group)
    if (!is.null(covar)) dat <- data.frame(dat, covar)
    pval <- anova(lm(y ~ ., data = dat), lm(y ~ . - cond, data = dat))$Pr[2]
    avgs <- tapply(p, m$group, mean)
    c(pval = pval, delta = unname(avgs[cmp$case] - avgs[cmp$ctrl]))   # change in % share
  })))
  res$padj <- p.adjust(res$pval, method = "BH")
  res %>% rownames_to_column("transcript_id") %>%
    mutate(gene_id = tx2gene$gene_id[match(transcript_id, tx2gene$transcript_id)],
           symbol  = sym[gene_id])
}

res_dtu <- lapply(names(comparisons), function(n) dtu_test(comparisons[[n]], res_DE[[n]]$gene))
names(res_dtu) <- names(comparisons)

dpct_cut <- 10   # at least 10 percentage points change in isoform share
sapply(res_dtu, function(r) c(isoforms_tested = nrow(r),
                              genes_tested    = length(unique(r$gene_id)),
                              DTU_isoforms    = sum(r$padj < padj_cut & abs(r$delta) > dpct_cut),
                              DTU_genes       = length(unique(r$gene_id[r$padj < padj_cut & abs(r$delta) > dpct_cut]))))

In [ ]:
# Volcano: change in isoform share vs significance
options(repr.plot.width = 7, repr.plot.height = 5)
for (n in names(comparisons)) {
  r <- res_dtu[[n]]
  r$DTU <- r$padj < padj_cut & abs(r$delta) > dpct_cut
  top <- rank(r$padj, ties.method = "first") <= 15
  r$label <- ifelse(r$DTU & top, r$symbol, NA)
  print(ggplot(r, aes(delta, -log10(padj), colour = DTU, label = label)) +
    geom_point(size = 0.8) +
    geom_text_repel(size = 3, max.overlaps = 30, na.rm = TRUE, show.legend = FALSE) +
    geom_vline(xintercept = c(-dpct_cut, dpct_cut), col = "#303030", linetype = "dotted") +
    geom_hline(yintercept = -log10(padj_cut), col = "#303030", linetype = "dotted") +
    scale_colour_manual(values = c(`FALSE` = "#909090", `TRUE` = "darkorange")) +
    labs(x = "change in isoform share (percentage points, case - control)", y = "-log10(padj)",
         title = paste0("Isoform usage: ", comparisons[[n]]$case, " vs ", comparisons[[n]]$ctrl,
                        " – ", length(unique(r$gene_id[r$DTU])), " genes")) +
    theme_bw())
}

In [ ]:
# Isoform shares of the top DTU gene per comparison: one bar per sample
plot_isoforms <- function(n) {
  r <- res_dtu[[n]]
  g <- r$gene_id[order(r$padj)][1]
  m <- get_samples(comparisons[[n]])
  m <- m[order(m$group != comparisons[[n]]$ctrl), ]                  # controls first
  tx <- tx2gene$transcript_id[tx2gene$gene_id == g]
  tx <- tx[rowMeans(iso_pct[tx, m$SRR, drop = FALSE]) >= 5]           # isoforms with >= 5 % share
  d <- data.frame(transcript = rep(tx, times = nrow(m)),
                  sample = rep(paste(m$group, substr(m$SRR, 9, 11)), each = length(tx)),
                  pct = as.vector(iso_pct[tx, m$SRR]))
  d$sample <- factor(d$sample, levels = unique(d$sample))
  ggplot(d, aes(sample, pct, fill = transcript)) +
    geom_col() +
    labs(x = NULL, y = "isoform share (%)",
         title = paste0(n, ": ", sym[g], " (gene TPM ", round(mean(expr[g, m$SRR])),
                        ", best isoform padj ", signif(min(r$padj), 2), ")")) +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

options(repr.plot.width = 8, repr.plot.height = 4.5)
for (n in names(comparisons)) print(plot_isoforms(n))

In [ ]:
# Save the isoform-usage tables
for (n in names(comparisons))
  write.csv(res_dtu[[n]], file.path(out, paste0("isoform_usage_", n, ".csv")), row.names = FALSE)